In [1]:
import os
import tempfile

from redislite.falkordb_client import (
    FalkorDB,  # embedded FalkorDB: pip install falkordblite
)

import opencyver
from opencyver import PropertiesValidator, SchemaValidator, SyntaxValidator

In [2]:
print(opencyver.__version__)

3.0.1


## Movies database
This notebook uses a small Movies graph, modelled on the standard Neo4j Movies example dataset used in tutorials. The notebook creates it in an embedded [FalkorDB](https://www.falkordb.com/) (FalkorDBLite), so nothing needs to be installed or started.

The Movies graph is a small sample movie graph useful for learning Cypher queries and graph concepts.
It models movies, people (actors / directors), and simple relationships like :ACTED_IN, :DIRECTED, and :PRODUCED.

### Schema of the movie graph database

![Movie DB](images/movie_graph.jpg)


In [3]:
# Start an embedded FalkorDB (FalkorDBLite) and load the Movies graph. No server is needed.
# To use a FalkorDB server instead:  driver = falkordb.FalkorDB(host="localhost", port=6379)
MOVIES = """
CREATE (keanu:Person {name: 'Keanu Reeves', born: 1964}),
       (carrie:Person {name: 'Carrie-Anne Moss', born: 1967}),
       (lana:Person {name: 'Lana Wachowski', born: 1965}),
       (lilly:Person {name: 'Lilly Wachowski', born: 1967}),
       (joel:Person {name: 'Joel Silver', born: 1952}),
       (tom:Person {name: 'Tom Hanks', born: 1956}),
       (jack:Person {name: 'Jack Nicholson', born: 1937}),
       (jessica:Person {name: 'Jessica Thompson'}),
       (angela:Person {name: 'Angela Scope'}),
       (matrix:Movie {title: 'The Matrix', released: 1999, tagline: 'Welcome to the Real World'}),
       (cloud:Movie {title: 'Cloud Atlas', released: 2012, tagline: 'Everything is connected'}),
       (asgood:Movie {title: 'As Good as It Gets', released: 1997, tagline: 'A comedy from the heart'}),
       (keanu)-[:ACTED_IN {roles: ['Neo']}]->(matrix),
       (carrie)-[:ACTED_IN {roles: ['Trinity']}]->(matrix),
       (tom)-[:ACTED_IN {roles: ['Zachry']}]->(cloud),
       (jack)-[:ACTED_IN {roles: ['Melvin Udall']}]->(asgood),
       (lana)-[:DIRECTED]->(matrix), (lilly)-[:DIRECTED]->(matrix),
       (lana)-[:WROTE]->(matrix), (lilly)-[:WROTE]->(matrix),
       (lana)-[:WROTE]->(cloud),
       (joel)-[:PRODUCED]->(matrix),
       (jessica)-[:REVIEWED {summary: 'An amazing journey', rating: 95}]->(cloud),
       (angela)-[:FOLLOWS]->(jessica)
"""

driver = FalkorDB(os.path.join(tempfile.mkdtemp(), "movies.db"))
database_name = "movies"  # in FalkorDB the database name is the name of the graph
driver.select_graph(database_name).query(MOVIES)
print("Graph", repr(database_name), "loaded.")

Graph 'movies' loaded.


---

## 🤔 Before You Dive Into Cypher...

Before running Cypher queries, ask yourself:

- 🤖 **LLMs are generating Cypher queries. Can we trust them?**
- 🌀 **What happens when they hallucinate?**
- 🧩 **Do they understand the schema, properties, and syntax of Cypher queries?**

💡 **In short:** The answer to all three comes down to **proper validation**.  
That’s where <span style="font-size:24px; color:#FF9800; font-weight:bold;">OpenCyVer</span> comes in!

<div align="center">
    <img src="images/logoCyVer.jpg" alt="OpenCyVer" width="300">
</div>

<span style="font-size:18px">
OpenCyVer is an open-source Python library that validates Cypher queries against your FalkorDB Knowledge Graph schema.  
It ensures <b>correctness in syntax, schema validity, and property access</b>.
</span>

<details>
<summary><b>✨ Why use OpenCyVer?</b></summary>

- 🛡️ <b>Trust LLM-generated Cypher:</b> Validate queries before execution.
- 🧠 <b>Prevent Hallucinations:</b> Catch schema and property mismatches.
- 🏗️ <b>Schema Awareness:</b> Ensure queries align with your graph’s structure.

</details>

---

# ✨ OpenCyVer (v3 Release)

OpenCyVer brings **three powerful classes** to validate Cypher queries—ensuring your graph work is robust, safe, and schema-aware.  
But first, let's clarify the core building blocks of Cypher regex patterns:

---

<div style="font-size:18px; font-weight:bold; color:#43A047;">🟢 Node Patterns</div>

A <b>node pattern</b> in Cypher represents an entity <code>(n:Label{props})</code> and can include:
- An optional variable (e.g., <code>n</code>)
- One or more labels (e.g., <code>n:Label</code>, <code>n:Label1:Label2</code>)
- Optional properties (e.g., <code>n:Label {key: value}</code>)

<details>
<summary>💡 <b>Examples</b></summary>

- <code>(n:Person)</code> — Node with label <b>Person</b>
- <code>(n:Person:Director)</code> — Node with both labels
- <code>(n:Movie {title: "Inception"})</code> — Node with property
</details>

---

<div style="font-size:18px; font-weight:bold; color:#1E88E5;">🔗 Relationship Patterns</div>

A <b>relationship pattern</b> in Cypher is written as <code>[r:REL_TYPE{props}]</code> and can include:
- An optional variable (e.g., <code>r</code>)
- One or more relationship types (e.g., <code>:REL_TYPE1</code>, <code>:REL_TYPE1|REL_TYPE2</code>)
- Variable length (e.g., <code>*</code>, <code>*1..3</code>)
- Optional properties (e.g., <code>r:REL_TYPE1{key: value}</code>)

<details>
<summary>💡 <b>Examples</b></summary>

- <code>[r:ACTED_IN]</code> — Relationship of type <b>ACTED_IN</b>
- <code>[:DIRECTED|PRODUCED]</code> — Either type
- <code>[r:ACTED_IN*1..3]</code> — Variable length
- <code>[r {role: "Trinity"}]</code> — With property
</details>

---

<div style="font-size:18px; font-weight:bold; color:#FBC02D;">🟢➡️🟢 Path Patterns</div>

A <b>path pattern</b> combines node and relationship patterns, with optional direction:

- <code>(a:Person)-[r:ACTED_IN]->(m:Movie)</code>
- <code>(n)-[:FRIEND*2..5]-(m)</code>

<details>
<summary>💡 <b>What does this mean?</b></summary>

A path pattern describes how nodes are connected—by relationships, with direction (<code>-></code> or <code>&lt;-</code>) or undirected (<code>--</code>).
</details>

---

> 📝 <b>With OpenCyVer, you can validate all these patterns for syntax, schema, and property correctness—before running your queries!</b>

## 🚀 Main Classes of <span style="color:#FF9800; font-weight:bold">OpenCyVer</span>

OpenCyVer provides three core validation classes to ensure your Cypher queries are robust, safe, and schema-aware:

---

### 🟠 <span style="color:#FF9800; font-weight:bold">SyntaxValidator</span>
- **Purpose:** Validates Cypher queries for correct syntax.
- **Features:** Detects any syntax errors in the provided Cypher query and also conflicting labels in nodes and relationships.
- **Why it matters:** Prevents malformed queries before they reach your database.

---

### 🟠 <span style="color:#FF9800; font-weight:bold">SchemaValidator</span>
- **Purpose:** Ensures Cypher queries conform to your knowledge graph’s schema.
- **Features:** Checks that node, relationship, and path patterns in queries match the expected schema structure.
- **Why it matters:** Maintains consistency and prevents structural errors.

---

### 🟠 <span style="color:#FF9800; font-weight:bold">PropertiesValidator</span>
- **Purpose:** Ensures only valid properties are accessed in queries.
- **Features:** Validates property access for nodes and relationships based on their labels or types.
- **Why it matters:** Maintains consistency and prevents structural errors.

---

📖 **Documentation:**  
Dive deeper into each validator in the [OpenCyVer documentation](https://github.com/Killeroid/OpenCyVer/tree/main/docs) on GitHub.

---

# Let's test OpenCyVer

In the below sections, we test a set of Cypher queries against each validator - one written correctly and others intentionally broken - to showcase how OpenCyVer detects and explains errors.

# Syntax Validator

### 🛠️ Define Example Cypher Queries for Syntax Validation

Each option of the provided cypher queries showcase a different error regarding the `SyntaxValidator`:

- **valid_query**: A correct Cypher query.
- **closing_parenthesis_query**: Missing a closing parenthesis (syntax error).
- **not_param_provided_query**: Uses a parameter that is not provided (common mistake).
- **unsupported_query**: Uses Cypher that FalkorDB does not support (here `&` between relationship types).
- **arithemtic_error_query**: Contains an explicit division by zero.
- **conflicting_label_query**: Uses the same variable for nodes with different labels (can cause conflicts).

In [4]:
valid_query = "MATCH (p:Person)-[r:ACTED_IN]->(m:Movie) RETURN p.name"
closing_parenthesis_query = "MATCH (p:Person-[r:ACTED_IN]->(m:Movie) RETURN p, m"
not_param_provided_query = "MATCH (p:Person{name:$prop_name}) RETURN p"
unsupported_query = "MATCH (p:Person)-[r:ACTED_IN&FOLLOWS]->(m:Movie) RETURN p, m"
arithemtic_error_query = "MATCH (p:Person) RETURN count(p)/0"
conflicting_label_query = "MATCH (p:Person) MATCH (p:Movie) RETURN p LIMIT 10"

# Step 2: Put them in a dictionary with descriptions
options = {
    1: ("Valid query", valid_query),
    2: ("Missing closing parenthesis", closing_parenthesis_query),
    3: ("Not provided parameter", not_param_provided_query),
    4: ("Unsupported Cypher (& between relationship types)", unsupported_query),
    5: ("Arithmetic error (division by zero)", arithemtic_error_query),
    6: ("Conflicting labels", conflicting_label_query),
}

# Step 3: Print a menu
print("Select a query you want to test with the Syntax validator:")
for key, (desc, _) in options.items():
    print(f"{key}. {desc}")

# Step 4: Choose the query to test: change this number (1-6) and run the cells below again
choice = 6
syntax_query = options[choice][1]


# Example function that uses the chosen variable
def use_global():
    print(f"Selected: \n  Candidate Query: {syntax_query} \n --> {options[choice][0]} ")


# Call the function
use_global()

Select a query you want to test with the Syntax validator:
1. Valid query
2. Missing closing parenthesis
3. Not provided parameter
4. Unsupported Cypher (& between relationship types)
5. Arithmetic error (division by zero)
6. Conflicting labels
Selected: 
  Candidate Query: MATCH (p:Person) MATCH (p:Movie) RETURN p LIMIT 10 
 --> Conflicting labels 


### 🧩 Let's test SyntaxValidator

1. **Initialize the validator:**  
   Create a `SyntaxValidator` object, passing your FalkorDB client (`falkordb.FalkorDB`, FalkorDBLite, or a `falkordb.Graph`). You can also enable advanced checks, such as for multi-labeled nodes.

2. **Validate your query:**  
   Call `.validate(query, database_name=...)` (`database_name` is the graph name) with your Cypher query.  
   The method returns:
   - `is_valid`: `True` if the query is syntactically correct, `False` otherwise.
   - `syntax_metadata`: A list of dictionaries containing as key error codes and as value the descriptions explaining any issues found.


In [5]:
# Initialize the syntax validator
syntax_validator = SyntaxValidator(driver, check_multilabeled_nodes=True)

# Validate the query's syntax using OpenCyVer
is_valid, syntax_metadata = syntax_validator.validate(
    syntax_query, database_name=database_name
)

# Output whether the syntax is valid (expected: False)
print(f"Syntax Valid: {is_valid}")

# Print detailed information about the syntax errors detected
print("Syntax Error Metadata:")
for metadata in syntax_metadata:
    print(f"{metadata['code']}: \n {metadata['description']}")

Syntax Valid: False
Syntax Error Metadata:
ConflictingVariablesError: 
 Variable 'p' has conflicting labels: Person, Movie


# Schema Validator

This section includes Cypher queries with incorrect node labels, relationship types, or invalid paths.

A powerful feature of OpenCyVer is **path-pattern inference**:  
> OpenCyVer infers missing node labels or relationship types by matching query structures to known schema path patterns.

### 🛠️ Define Example Cypher Queries for Schema Validation

Each option of the provided cypher queries showcase a different error regarding the `SchemaValidator`:

- **valid_query**: A correct Cypher query.
- **path_inference_query**: If a node label or relationship type is missing in a path, OpenCyVer will infer it from the schema.In this query, the label of variable m is missing. The extract method shows that m label is inferred from the KG schema.
- **multiple_inference_path_available_query**: If a node label or relationship type is missing in a path, OpenCyVer will infer it from the schema. In this query, multiple path inferences are possible. Run the Schema Validator execution cell to view the different extracted paths.
- **unknown_label_type_query**: The provided query uses a node label or relationship type not present in the schema (e.g., typo).
- **unknown_path_query**: This query attempts to connect two node labels with a disallowed relationship.
- **wrong_direction_path_query**: This query uses a valid relationship between two nodes but with wrong direction.
- **variable_length_path_query**: This query uses variable length in the relationship that does not exist.

In [6]:
valid_query = "MATCH (p:Person)-[r:ACTED_IN]->(m:Movie) RETURN p.name"
path_inference_query = (
    "MATCH (g:Person)-[:FOLLOWS]->(t:Person)-[:REVIEWED]->(m) RETURN *"
)
multiple_inference_path_available_query = "MATCH (g:Person)-[]->(t:Movie) RETURN *"
unknown_label_type_query = "MATCH (p:Persons)-[:WROTEs]->(m:Movie) RETURN p"
unknown_path_query = "MATCH (p:Person)-[:FOLLOWS]->(m:Movie) RETURN p LIMIT 4"
wrong_direction_path_query = "MATCH (p:Person)<-[:PRODUCED]-(M:Movie) RETURN p LIMIT 10"
variable_length_path_query = (
    "MATCH (p:Person)-[:FOLLOWS*3..4]->(m:Person) RETURN * LIMIT 5"
)


# Step 2: Put them in a dictionary with descriptions
options = {
    1: ("Valid query", valid_query),
    2: (
        "Path inference (missing node label, one possible inference)",
        path_inference_query,
    ),
    3: (
        "Path inference (missing node label, multiple possible inferences)",
        multiple_inference_path_available_query,
    ),
    4: ("Unknown node label/relationship type", unknown_label_type_query),
    5: ("Unknown path", unknown_path_query),
    6: ("Wrong direction in path", wrong_direction_path_query),
    7: ("Variable length path", variable_length_path_query),
}

# Step 3: Print a menu
print("Select a query you want to test with the Schema validator:")
for key, (desc, _) in options.items():
    print(f"{key}. {desc}")

# Step 4: Choose the query to test: change this number (1-7) and run the cells below again
choice = 2
schema_query = options[choice][1]


# Example function that uses the chosen variable
def use_global():
    print(f"Selected: \n  Candidate Query: {schema_query} \n --> {options[choice][0]} ")


# Call the function
use_global()

Select a query you want to test with the Schema validator:
1. Valid query
2. Path inference (missing node label, one possible inference)
3. Path inference (missing node label, multiple possible inferences)
4. Unknown node label/relationship type
5. Unknown path
6. Wrong direction in path
7. Variable length path
Selected: 
  Candidate Query: MATCH (g:Person)-[:FOLLOWS]->(t:Person)-[:REVIEWED]->(m) RETURN * 
 --> Path inference (missing node label, one possible inference) 


### 🧩 How to Use the SchemaValidator

1. **Initialize the validator:**  
   Create a `SchemaValidator` object, passing your FalkorDB client.

2. **Extract patterns:**  
   Use `.extract(query, database_name=...)` to see which extracted node patterns, relationship patterns, and path patterns your query uses. This helps you understand how your query maps to the schema.

3. **Validate your query:**  
   Call `.validate(query, database_name=...)` to check if the query structure matches your graph schema.  
   The method returns:
   - `schema_score`: The schema score is computed as a weighted average of the valid extracted patterns:

   ![Schema Score](images/Schema_score.jpg)

   where $w_1, w_2, w_3$ are weighting factors that depend on the extracted components. Read more about the schema formula in the docstring of `SchemaValidator.validate`.

   - `schema_metadata`: A list of dictionaries containing as key error codes and as value the descriptions explaining any issues found.

In [7]:
schema_validator = SchemaValidator(driver)

# Extraction
print("--" * 50)
extracted_node_labels, extracted_rel_labels, extracted_paths = schema_validator.extract(
    schema_query, database_name=database_name
)
print(f"Extracted Node patterns : {extracted_node_labels}")
print(f"Extracted Relationships Patterns : {extracted_rel_labels}")
print(f"Extracted Path patterns : {extracted_paths}")
# Validation
print("--" * 50)
schema_score, schema_metadata = schema_validator.validate(
    schema_query, database_name=database_name
)
print(f"Schema Validation Score: {schema_score}")
print("Schema Validation Metadata:")
for metadata in schema_metadata:
    print(f"{metadata['code']}: \n {metadata['description']}")

----------------------------------------------------------------------------------------------------
Extracted Node patterns : ['(:Person)']
Extracted Relationships Patterns : ['()-[:FOLLOWS]-()', '()-[:REVIEWED]-()']
Extracted Path patterns : ['(:Person)-[:FOLLOWS]->(:Person)', '(:Person)-[:REVIEWED]->(:Movie)']
----------------------------------------------------------------------------------------------------
Schema Validation Score: 1.0
Schema Validation Metadata:


# Properties Validator

### 🛠️ Define Example Cypher Queries for Properties Validation

Each option of the provided cypher queries showcase a different error regarding the `PropertiesValidator`. 

The path-pattern inference is also available in the Properties Validator, as shown below.


- **valid_query**: A correct Cypher query.
- **unknown_prop_query**: The Cypher query accesses an unknown property.
- **invalid_prop_by_label_query**: The Cypher query accesses an invalid property for the node label or relationship type.
- **inference_prop_query**: Property is accessed on a node with missing label in the provided query; OpenCyVer will infer this label and will validate if the property belongs to this node.
- **valid_rel_prop_query**: Accesses a property of a relationship that has it.
- **invalid_rel_prop_query**: Accesses a property that the relationship type does not have.
- **strict_query**: An example to test the *strict* parameter in Properties Validator. Use True and False to see the difference.

In [8]:
valid_query = "MATCH (p:Movie) RETURN p.title "
unknown_prop_query = "MATCH (g:Person) RETURN g.age"
invalid_prop_by_label_query = "MATCH (g:Movie) RETURN g.name"
inference_prop_query = "MATCH (p:Person)-[:WROTE]->(m) RETURN m.released"
valid_rel_prop_query = "MATCH (p:Person)-[r:REVIEWED]->(m:Movie) RETURN r.rating"
invalid_rel_prop_query = "MATCH (p:Person)-[r:ACTED_IN]->(m:Movie) RETURN r.rating"
strict_query = "MATCH (p) RETURN p.title,p.tagline,p.name LIMIT 4"


# Step 2: Put them in a dictionary with descriptions
options = {
    1: ("Valid query", valid_query),
    2: ("Unknown property", unknown_prop_query),
    3: ("Invalid property by label", invalid_prop_by_label_query),
    4: ("Inference of node label based on the accessed property", inference_prop_query),
    5: ("Relationship property (valid)", valid_rel_prop_query),
    6: ("Invalid property by relationship type", invalid_rel_prop_query),
    7: (
        "Strict mode: multiple properties, some may not exist (Please use strict = True/False to see the differences)",
        strict_query,
    ),
}

# Step 3: Print a menu
print("Select a query you want to test with the Properties validator:")
for key, (desc, _) in options.items():
    print(f"{key}. {desc}")

# Step 4: Choose the query to test: change this number (1-7) and run the cells below again
choice = 4
props_query = options[choice][1]


# Example function that uses the chosen variable
def use_global():
    print(f"Selected: \n  Candidate Query: {props_query} \n --> {options[choice][0]} ")


# Call the function
use_global()

Select a query you want to test with the Properties validator:
1. Valid query
2. Unknown property
3. Invalid property by label
4. Inference of node label based on the accessed property
5. Relationship property (valid)
6. Invalid property by relationship type
7. Strict mode: multiple properties, some may not exist (Please use strict = True/False to see the differences)
Selected: 
  Candidate Query: MATCH (p:Person)-[:WROTE]->(m) RETURN m.released 
 --> Inference of node label based on the accessed property 


### 🧩 How to Use the PropertiesValidator

1. **Initialize the validator:**  
   Create a `PropertiesValidator` object, passing your FalkorDB client.

2. **Extract properties:**  
   Use `.extract(query, strict=False, database_name=...)` to see which properties are accessed by variables and labels in your query. It returns two mappings:

   - **Variables-Properties Mapping:**  
      Shows which properties are accessed by each variable in the query.  
      ```
      Var_x → [property1, property2]
      ```

   - **Labels-Properties Mapping:**  
      Shows which properties are accessed for each node label or relationship type (including inferred labels/types).  
      ```
      nodeLabel / relType → [property1, property2]
      ```

3. **Validate your query:**  
   Call `.validate(query, database_name=..., strict=False)` to check if the properties accessed in your query are valid for the corresponding nodes or relationships.  
   The method returns:
   - `props_score`: A score in [0, 1] indicating the percentage of correctly accessed properties:

   ![Properties Score](images/Properties_score.jpg)

   - `props_metadata`: A list of dictionaries containing as key error codes and as value the descriptions explaining any issues found.

In [9]:
props_validator = PropertiesValidator(driver)

# Set strict parameter to True or False to see the difference
strict = False

# Extraction
print("--" * 50)
variables_properties, labels_properties = props_validator.extract(
    props_query, strict=strict, database_name=database_name
)
print(f"Accessed properties by variables: {variables_properties}")
print(f"Accessed properties by labels (including inferred): {labels_properties}")

# Validation
print("--" * 50)
props_score, props_metadata = props_validator.validate(
    props_query, database_name=database_name, strict=strict
)
print(f"Properties Validation Score: {props_score}")
print("Properties Validation Metadata:")
for metadata in props_metadata:
    print(f"{metadata['code']}: \n {metadata['description']}")

----------------------------------------------------------------------------------------------------
Accessed properties by variables: {'m': ['released']}
Accessed properties by labels (including inferred): {'Movie': ['released']}
----------------------------------------------------------------------------------------------------
Properties Validation Score: 1.0
Properties Validation Metadata:


## 📊 **Tip:**  
> Always start with the **SyntaxValidator** before using the **Schema** and **Properties** validators. Even a small syntax error can cause misleading results in downstream validation. Ensuring syntactic correctness first helps you avoid wasted time and confusion. Building on this principle, we define the KG Valid Query metric to systematically evaluate generated Cypher queries for knowledge graphs. 

A query is considered valid only if it satisfies all three checks:

1. SyntaxValidator.is_valid= True → the query is syntactically correct.
2. SchemaValidator.score = 1 → the query respects the graph schema.
3. PropertiesValidator.score = 1 or None → the query accesses only valid properties.

The **KG Valid Query** is a recommended metric, as described in [Decoding the Mystery: How Can LLMs Turn Text Into Cypher in Complex Knowledge Graphs?](https://ieeexplore.ieee.org/document/10990239):

 ![KG Valid Query](images/KG_Valid_Query.jpg)